# 07 — Strict sample and main analysis

Demo-mode analysis only. Formal definitions: 12 classes, country n ≥ 500 within the strict paper sample, US and male references, JSD-computable papers with at least two valid reviews, and the source-code predicate `n_authors <= 50`. Papers with more than 50 authors are excluded; exactly 50 authors are retained. No real data are bundled here. Future use requires synthetic data or authorized researcher-provided row-level data.

The reduced threshold is used only for the synthetic demonstration. The formal study threshold is 500 papers.

Review-level aspect-ratio OLS controls: `publication_year + C(subject_primary) + log_review_word_count + log_n_authors`; cluster-robust covariance by `paper_code`. Paper-level mean pairwise JSD OLS controls: `publication_year + C(subject_primary) + n_reviews_valid_for_jsd + log_aspect_total_count + log_n_authors`; HC3 covariance. Both use country, gender, and combined specifications with explicit US/male references.

SOURCE_LINEAGE_LITERAL: `main_analysis_strict_sample.ipynb`, source cells 0–9; cell 9 contributes aggregate figure data only. Run notebooks 01–06 first. Outputs stay under `results/demo/strict`.


In [ ]:
from __future__ import annotations

import re
import warnings
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from pathlib import Path
import sys
import yaml

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "project_paths.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "project_paths.py").exists():
    raise FileNotFoundError("Run from the release root or its notebooks directory.")
sys.path.insert(0, str(ROOT))
from src.project_paths import project_path

with (ROOT / "config" / "config.example.yaml").open(encoding="utf-8") as fh:
    CONFIG = yaml.safe_load(fh)
MODE = CONFIG.get("runtime", {}).get("mode", "demo")
if MODE not in ('demo',):
    raise ValueError("This notebook requires runtime.mode in: demo")
ANALYSIS = CONFIG["analysis"]
if int(ANALYSIS["formal_country_min_papers"]) != 500:
    raise ValueError("The formal country threshold must remain 500.")
if int(ANALYSIS["large_team_cutoff"]) != 50:
    raise ValueError("The formal large-team cutoff must remain 50.")
if ANALYSIS["country_reference"] != "US" or ANALYSIS["gender_reference"] != "male":
    raise ValueError("Formal model references must remain US and male.")
WORK_DIR = project_path("data/work")
OUT = project_path("results/demo/strict")
OUT_SAMPLES = OUT / "samples"
OUT_TABLES = OUT / "tables"
OUT_FIG = OUT / "figures"
for directory in (OUT_SAMPLES, OUT_TABLES):
    directory.mkdir(parents=True, exist_ok=True)
ASPECT_CLASSES = [
    "Methodology",
    "Results",
    "Experiments and Evaluation",
    "Data and Datasets",
    "Figures and Tables",
    "Writing, Language and Terminology",
    "Theory and Models",
    "Comparison and Context",
    "Paper Assessment",
    "Novelty and Impact",
    "Validity, Rigor and Quality",
    "Limitations",
]

ASPECT_SHORT = {
    "Methodology": "Methodology",
    "Results": "Results",
    "Experiments and Evaluation": "Experiments & evaluation",
    "Data and Datasets": "Data & datasets",
    "Figures and Tables": "Figures & tables",
    "Writing, Language and Terminology": "Writing & language",
    "Theory and Models": "Theory & models",
    "Comparison and Context": "Comparison & context",
    "Paper Assessment": "Paper assessment",
    "Novelty and Impact": "Novelty & impact",
    "Validity, Rigor and Quality": "Validity & rigor",
    "Limitations": "Limitations",
}

COUNTRY_COL = "first_author_country_group_n500_strict"
GENDER_COL = "first_author_gender_confident"
JSD_COL = "mean_pairwise_jsd"
COUNTRY_REFERENCE = ANALYSIS["country_reference"]
GENDER_REFERENCE = ANALYSIS["gender_reference"]
COUNTRY_MIN_PAPERS = int(CONFIG["demo"]["country_min_papers"])  # DEMO ONLY; formal value remains 500
LARGE_TEAM_CUTOFF = int(ANALYSIS["large_team_cutoff"])  # source executable predicate is n_authors <= 50

# Color maps. Figure text is English; no figure titles are used.
DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "blue_white_orange",
    ["#4C78A8", "#F7F7F7", "#F58518"],
    N=256,
)

In [ ]:
# =========================================================
# 1. Utility functions
# =========================================================

def save_table(df: pd.DataFrame, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix == ".parquet":
        df.to_parquet(path, index=False)
    elif path.suffix == ".csv":
        df.to_csv(path, index=False, encoding="utf-8-sig")
    else:
        raise ValueError(f"Unsupported output format: {path}")
    print(f"Saved: {path} | rows={len(df):,}, cols={df.shape[1]}")


def save_fig(name: str) -> None:
    path = OUT_FIG / name
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print(f"Saved: {path}")


def read_preferred(paths: Sequence[Path]) -> pd.DataFrame:
    for p in paths:
        if p.exists():
            print(f"Reading: {p}")
            return pd.read_parquet(p)
    raise FileNotFoundError("None of the candidate files exists:\n" + "\n".join(map(str, paths)))


def clean_value(x, unknown: str = "Unknown") -> str:
    if pd.isna(x):
        return unknown
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "none", "null", "na"}:
        return unknown
    return x


def clean_gender(x) -> str:
    x = clean_value(x)
    x = x.lower()
    if x in {"male", "m"}:
        return "male"
    if x in {"female", "f", "woman", "women"}:
        return "female"
    return "Unknown"


def safe_log1p(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    s = s.where(s >= 0)
    return np.log1p(s)


def coerce_for_statsmodels(
    df: pd.DataFrame,
    numeric_cols: list[str] | None = None,
    categorical_cols: list[str] | None = None,
    group_col: str | None = None,
) -> pd.DataFrame:
    """Convert pandas nullable dtypes to NumPy/Python dtypes before statsmodels formulas."""
    out = df.copy()
    numeric_cols = numeric_cols or []
    categorical_cols = categorical_cols or []

    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")

    for col in categorical_cols:
        if col in out.columns:
            out[col] = out[col].astype("object").where(out[col].notna(), "Unknown").astype(str)

    if group_col is not None and group_col in out.columns:
        out[group_col] = out[group_col].astype("object").astype(str)

    for col in out.columns:
        if pd.api.types.is_extension_array_dtype(out[col].dtype):
            if pd.api.types.is_numeric_dtype(out[col].dtype):
                out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")
            elif pd.api.types.is_bool_dtype(out[col].dtype):
                out[col] = out[col].astype("float64")
            else:
                out[col] = out[col].astype("object").where(out[col].notna(), "Unknown").astype(str)

    return out


def prepare_subject_primary(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "subject_primary" in out.columns:
        out["subject_primary"] = out["subject_primary"].apply(clean_value)
    elif "subject_codes_final" in out.columns:
        out["subject_primary"] = (
            out["subject_codes_final"]
            .fillna("Unknown")
            .astype(str)
            .str.split("|")
            .str[0]
            .str.strip()
            .replace({"": "Unknown", "nan": "Unknown", "None": "Unknown"})
        )
    else:
        out["subject_primary"] = "Unknown"
    return out


def add_strict_country_group(df: pd.DataFrame, country_col: str = "first_author_country") -> pd.DataFrame:
    """Recompute n>=500 country grouping on the strict main sample, not on older no-capped samples."""
    out = df.copy()
    if country_col not in out.columns:
        raise KeyError(f"Missing country column: {country_col}")

    country = out[country_col].apply(clean_value)
    counts = country[country != "Unknown"].value_counts()
    keep = counts[counts >= COUNTRY_MIN_PAPERS].index.tolist()

    def mapper(x: str) -> str:
        if x == "Unknown":
            return "Unknown"
        if x in keep:
            return x
        return "Other"

    out[COUNTRY_COL] = country.map(mapper)

    country_keep = pd.DataFrame({
        "country": counts.index,
        "n_papers": counts.values,
        "kept_as_separate_group": [c in keep for c in counts.index],
    })
    save_table(country_keep, OUT_TABLES / "country_grouping_n500_on_strict_sample.csv")
    return out


def harmonize_background(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = prepare_subject_primary(out)

    numeric_cols = [
        "publication_year",
        "n_authors",
        "n_reviews",
        "n_reviews_valid_for_jsd",
        "aspect_total_count",
        "mean_pairwise_jsd",
        "centroid_jsd",
        "max_pairwise_jsd",
    ]
    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    if GENDER_COL in out.columns:
        out[GENDER_COL] = out[GENDER_COL].apply(clean_gender)
    else:
        out[GENDER_COL] = "Unknown"

    out["log_n_authors"] = safe_log1p(out["n_authors"])
    if "aspect_total_count" in out.columns:
        out["log_aspect_total_count"] = safe_log1p(out["aspect_total_count"])
    else:
        out["log_aspect_total_count"] = np.nan

    return out


def align_review_to_papers(review_df: pd.DataFrame, paper_df: pd.DataFrame) -> pd.DataFrame:
    """Restrict review-level data to the exact paper_code set used by the strict paper-level sample."""
    paper_cols = [
        "paper_code",
        COUNTRY_COL,
        GENDER_COL,
        "publication_year",
        "subject_primary",
        "n_authors",
        "log_n_authors",
    ]
    paper_cols = [c for c in paper_cols if c in paper_df.columns]

    # Drop stale background columns from review table to avoid suffix conflicts.
    drop_cols = [c for c in paper_cols if c != "paper_code" and c in review_df.columns]
    out = review_df.drop(columns=drop_cols, errors="ignore").copy()

    out = out[out["paper_code"].isin(paper_df["paper_code"])].copy()
    out = out.merge(paper_df[paper_cols], on="paper_code", how="left")

    for c in ASPECT_CLASSES + ["aspect_total_count"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0)

    return out


def sample_count(df: pd.DataFrame, label: str, unit: str = "paper") -> dict:
    row = {"sample": label}
    if "paper_code" in df.columns:
        row["n_papers"] = df["paper_code"].nunique()
    if "review_id" in df.columns:
        row["n_reviews"] = df["review_id"].nunique()
    elif "n_reviews" in df.columns:
        row["n_reviews"] = int(pd.to_numeric(df["n_reviews"], errors="coerce").fillna(0).sum())
    if unit:
        row["unit"] = unit
    return row


def summarize_group_size_paper(df: pd.DataFrame, group_col: str, prefix: str) -> pd.DataFrame:
    tab = (
        df.assign(_group=df[group_col].apply(clean_value))
        .groupby("_group", dropna=False)
        .agg(
            n_papers=("paper_code", "nunique"),
            n_reviews=("n_reviews", "sum") if "n_reviews" in df.columns else ("paper_code", "size"),
        )
        .reset_index()
        .rename(columns={"_group": group_col})
        .sort_values("n_papers", ascending=False)
    )
    tab["paper_share"] = tab["n_papers"] / tab["n_papers"].sum()
    save_table(tab, OUT_TABLES / f"descriptive_group_size_paper__{prefix}.csv")
    return tab


def summarize_group_size_review(df: pd.DataFrame, group_col: str, prefix: str) -> pd.DataFrame:
    tab = (
        df.assign(_group=df[group_col].apply(clean_value))
        .groupby("_group", dropna=False)
        .agg(
            n_papers=("paper_code", "nunique"),
            n_reviews=("review_id", "nunique"),
        )
        .reset_index()
        .rename(columns={"_group": group_col})
        .sort_values("n_reviews", ascending=False)
    )
    tab["review_share"] = tab["n_reviews"] / tab["n_reviews"].sum()
    save_table(tab, OUT_TABLES / f"descriptive_group_size_review__{prefix}.csv")
    return tab


In [ ]:
# =========================================================
# 2. Load data and build strict main sample
# =========================================================

paper_bg = read_preferred([
    WORK_DIR / "paper_background_features_v2.parquet",
])

review_counts = read_preferred([
    WORK_DIR / "review_aspect_class_counts_v2.parquet",
])

paper_jsd = read_preferred([
    WORK_DIR / "paper_aspect_disagreement_v2.parquet",
])

# If the JSD table does not already contain background variables, merge them here.
bg_cols = [
    "paper_code",
    "n_authors",
    "first_author_country",
    "corresponding_author_country",
    "dominant_country",
    "first_author_gender_confident",
    "last_author_gender_confident",
    "first_last_gender_pair",
    "publication_year",
    "subject_primary",
    "subject_codes_final",
    "exclude_large_team_ge_50",
    "large_team_ge_50",
]
bg_cols = [c for c in bg_cols if c in paper_bg.columns]
missing_bg_key_cols = [c for c in ["n_authors", "first_author_country", GENDER_COL] if c not in paper_jsd.columns]
if missing_bg_key_cols:
    paper_jsd = paper_jsd.drop(columns=[c for c in bg_cols if c != "paper_code" and c in paper_jsd.columns], errors="ignore")
    paper_jsd = paper_jsd.merge(paper_bg[bg_cols], on="paper_code", how="left")

# Harmonize paper table.
paper_jsd = harmonize_background(paper_jsd)

# Main paper sample: JSD-computable + has author background + excludes > 50-author teams.
paper_main = paper_jsd[
    paper_jsd[JSD_COL].notna()
    & (pd.to_numeric(paper_jsd["n_reviews_valid_for_jsd"], errors="coerce") >= 2)
    & paper_jsd["n_authors"].notna()
    & (paper_jsd["n_authors"] <= LARGE_TEAM_CUTOFF)
].copy()

paper_main = add_strict_country_group(paper_main, country_col="first_author_country")
paper_main = harmonize_background(paper_main)

# Important: do not use international collaboration variables in main analysis.
# Derived samples from the strict main sample.
paper_country = paper_main[paper_main[COUNTRY_COL] != "Unknown"].copy()
paper_gender = paper_main[paper_main[GENDER_COL].isin(["male", "female"])].copy()
paper_combined = paper_main[
    (paper_main[COUNTRY_COL] != "Unknown")
    & (paper_main[GENDER_COL].isin(["male", "female"]))
].copy()

# Align review-level records to the strict paper sample.
review_counts = review_counts[pd.to_numeric(review_counts["aspect_total_count"], errors="coerce").fillna(0) > 0].copy()
review_main = align_review_to_papers(review_counts, paper_main)
review_country = review_main[review_main[COUNTRY_COL] != "Unknown"].copy()
review_gender = review_main[review_main[GENDER_COL].isin(["male", "female"])].copy()
review_combined = review_main[
    (review_main[COUNTRY_COL] != "Unknown")
    & (review_main[GENDER_COL].isin(["male", "female"]))
].copy()

# Save samples for reuse.
save_table(paper_main, OUT_SAMPLES / "main_strict_paper_jsd_no_large50.parquet")
save_table(paper_country, OUT_SAMPLES / "main_strict_paper_country_known.parquet")
save_table(paper_gender, OUT_SAMPLES / "main_strict_paper_gender_known.parquet")
save_table(paper_combined, OUT_SAMPLES / "main_strict_paper_country_gender_known.parquet")
save_table(review_main, OUT_SAMPLES / "main_strict_review_aligned.parquet")
save_table(review_country, OUT_SAMPLES / "main_strict_review_country_known.parquet")
save_table(review_gender, OUT_SAMPLES / "main_strict_review_gender_known.parquet")
save_table(review_combined, OUT_SAMPLES / "main_strict_review_country_gender_known.parquet")

sample_summary = pd.DataFrame([
    sample_count(paper_main, "Main strict paper-level sample", unit="paper"),
    sample_count(review_main, "Main strict review-level sample aligned to paper sample", unit="review"),
    sample_count(paper_country, "Country analysis paper sample", unit="paper"),
    sample_count(review_country, "Country analysis review sample", unit="review"),
    sample_count(paper_gender, "Gender analysis paper sample", unit="paper"),
    sample_count(review_gender, "Gender analysis review sample", unit="review"),
    sample_count(paper_combined, "Country + gender paper sample", unit="paper"),
    sample_count(review_combined, "Country + gender review sample", unit="review"),
])
save_table(sample_summary, OUT_TABLES / "descriptive_main_sample_summary.csv")
print(sample_summary)

In [ ]:
# =========================================================
# 3. Descriptive statistics: CSV only, no plots
# =========================================================

summarize_group_size_paper(paper_main, COUNTRY_COL, "country_all_including_unknown")
summarize_group_size_paper(paper_main, GENDER_COL, "first_author_gender_all")
summarize_group_size_review(review_main, COUNTRY_COL, "country_all_including_unknown")
summarize_group_size_review(review_main, GENDER_COL, "first_author_gender_all")

# JSD descriptive statistics.
def summarize_jsd(df: pd.DataFrame, group_col: str | None = None, prefix: str = "overall") -> pd.DataFrame:
    if group_col is None:
        vals = df[JSD_COL].dropna()
        out = pd.DataFrame([{
            "group_col": "overall",
            "group": "overall",
            "n_papers": df["paper_code"].nunique(),
            "n_reviews": int(df["n_reviews"].sum()) if "n_reviews" in df.columns else np.nan,
            "mean_jsd": vals.mean(),
            "median_jsd": vals.median(),
            "sd_jsd": vals.std(),
            "q25_jsd": vals.quantile(0.25),
            "q75_jsd": vals.quantile(0.75),
        }])
    else:
        rows = []
        for group, g in df.assign(_group=df[group_col].apply(clean_value)).groupby("_group"):
            vals = g[JSD_COL].dropna()
            rows.append({
                "group_col": group_col,
                "group": group,
                "n_papers": g["paper_code"].nunique(),
                "n_reviews": int(g["n_reviews"].sum()) if "n_reviews" in g.columns else np.nan,
                "mean_jsd": vals.mean(),
                "median_jsd": vals.median(),
                "sd_jsd": vals.std(),
                "q25_jsd": vals.quantile(0.25),
                "q75_jsd": vals.quantile(0.75),
            })
        out = pd.DataFrame(rows).sort_values("n_papers", ascending=False)

    save_table(out, OUT_TABLES / f"descriptive_jsd_summary__{prefix}.csv")
    return out

summarize_jsd(paper_main, prefix="main_strict")
summarize_jsd(paper_country, COUNTRY_COL, "country_known")
summarize_jsd(paper_gender, GENDER_COL, "gender_known")

# Aspect distribution tables.
def aspect_overall_distribution(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    total = df[ASPECT_CLASSES].sum().sum()
    n_reviews = df["review_id"].nunique()
    n_papers = df["paper_code"].nunique()
    rows = []
    for cls in ASPECT_CLASSES:
        cnt = float(df[cls].sum())
        review_cover = int((df[cls] > 0).sum())
        paper_cover = int(df.loc[df[cls] > 0, "paper_code"].nunique())
        rows.append({
            "aspect_class": cls,
            "aspect_label": ASPECT_SHORT.get(cls, cls),
            "count": cnt,
            "share": cnt / total if total > 0 else np.nan,
            "n_reviews_cover": review_cover,
            "review_cover_ratio": review_cover / n_reviews if n_reviews > 0 else np.nan,
            "n_papers_cover": paper_cover,
            "paper_cover_ratio": paper_cover / n_papers if n_papers > 0 else np.nan,
            "n_reviews": n_reviews,
            "n_papers": n_papers,
        })
    out = pd.DataFrame(rows).sort_values("share", ascending=False)
    save_table(out, OUT_TABLES / f"descriptive_aspect_overall__{prefix}.csv")
    return out

aspect_overall_distribution(review_main, "main_strict")
aspect_overall_distribution(review_country, "country_known")
aspect_overall_distribution(review_gender, "gender_known")

In [ ]:
# =========================================================
# 4. Aspect distribution association tests
# =========================================================

def build_aspect_contingency(df: pd.DataFrame, group_col: str, exclude_groups: Iterable[str] | None = None) -> pd.DataFrame:
    temp = df.copy()
    temp[group_col] = temp[group_col].apply(clean_value)
    if exclude_groups is not None:
        temp = temp[~temp[group_col].isin(set(exclude_groups))].copy()
    contingency = temp.groupby(group_col)[ASPECT_CLASSES].sum().astype(float)
    contingency = contingency.loc[contingency.sum(axis=1) > 0]
    return contingency


def cramers_v_from_chi2(chi2: float, n: float, r: int, c: int) -> float:
    denom = n * min(r - 1, c - 1)
    if denom <= 0:
        return np.nan
    return float(np.sqrt(chi2 / denom))


def adjusted_standardized_residuals(observed: np.ndarray, expected: np.ndarray) -> np.ndarray:
    observed = np.asarray(observed, dtype=float)
    expected = np.asarray(expected, dtype=float)
    total = observed.sum()
    row_prop = observed.sum(axis=1, keepdims=True) / total
    col_prop = observed.sum(axis=0, keepdims=True) / total
    denom = np.sqrt(expected * (1 - row_prop) * (1 - col_prop))
    with np.errstate(divide="ignore", invalid="ignore"):
        residuals = (observed - expected) / denom
    return residuals


def chi_square_aspect_test(df: pd.DataFrame, group_col: str, prefix: str, exclude_groups: Iterable[str] | None = None) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    contingency = build_aspect_contingency(df, group_col, exclude_groups=exclude_groups)
    chi2, p, dof, expected = chi2_contingency(contingency.values)
    n = contingency.values.sum()
    v = cramers_v_from_chi2(chi2, n, contingency.shape[0], contingency.shape[1])

    summary = pd.DataFrame([{
        "group_col": group_col,
        "n_groups": contingency.shape[0],
        "n_aspects": contingency.shape[1],
        "n_aspect_mentions": n,
        "chi2": chi2,
        "dof": dof,
        "p_value": p,
        "cramers_v": v,
    }])

    expected_df = pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)
    resid_df = pd.DataFrame(
        adjusted_standardized_residuals(contingency.values, expected),
        index=contingency.index,
        columns=contingency.columns,
    )

    save_table(summary, OUT_TABLES / f"test_chi_square_aspect__{prefix}.csv")
    contingency.to_csv(OUT_TABLES / f"test_chi_square_aspect__{prefix}__observed.csv", encoding="utf-8-sig")
    expected_df.to_csv(OUT_TABLES / f"test_chi_square_aspect__{prefix}__expected.csv", encoding="utf-8-sig")
    resid_df.to_csv(OUT_TABLES / f"test_chi_square_aspect__{prefix}__adjusted_residuals.csv", encoding="utf-8-sig")

    return summary, contingency, resid_df

chi_square_aspect_test(review_country, COUNTRY_COL, "country", exclude_groups=["Unknown"])
chi_square_aspect_test(review_gender, GENDER_COL, "gender", exclude_groups=["Unknown"])

In [ ]:
# =========================================================
# 5. Review-level aspect ratio models: CSV results only
# =========================================================

def aspect_ratio_col(cls: str) -> str:
    """Use one consistent column-name rule for all aspect-ratio variables."""
    return f"ratio__{re.sub(r'[^A-Za-z0-9]+', '_', cls).strip('_')}"


def add_review_model_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add aspect-ratio outcomes and review-level controls to any review-level sample."""
    out = df.copy()
    aspect_total = pd.to_numeric(out["aspect_total_count"], errors="coerce")

    for cls in ASPECT_CLASSES:
        y = aspect_ratio_col(cls)
        if y not in out.columns:
            aspect_count = pd.to_numeric(out[cls], errors="coerce").fillna(0)
            out[y] = np.where(aspect_total > 0, aspect_count / aspect_total, np.nan)

    if "review_word_count" in out.columns:
        review_word_count = out["review_word_count"]
    else:
        review_word_count = pd.Series(0, index=out.index)

    out["log_review_word_count"] = safe_log1p(review_word_count)

    return out


def fit_review_aspect_ols(df: pd.DataFrame, predictor_type: str, formula_rhs: str, prefix: str) -> pd.DataFrame:
    rows = []
    temp = add_review_model_features(df)

    model_base_cols = [
        "paper_code",
        COUNTRY_COL,
        GENDER_COL,
        "publication_year",
        "subject_primary",
        "log_review_word_count",
        "log_n_authors",
    ]
    model_base_cols = [c for c in model_base_cols if c in temp.columns]

    for cls in ASPECT_CLASSES:
        y = aspect_ratio_col(cls)

        if y not in temp.columns:
            print(f"Missing outcome column after preprocessing: {y}")
            continue

        #model_cols = [y] + model_base_cols
        #model_df = temp[model_cols].dropna().copy()

        #if model_df.empty:
        #    continue

        #formula = f"Q('{y}') ~ {formula_rhs}"
        model_cols = [y] + model_base_cols
        model_df = temp[model_cols].dropna().copy()
        if model_df.empty:
            continue

        model_df = coerce_for_statsmodels(
            model_df,
            numeric_cols=[y, "publication_year", "log_review_word_count", "log_n_authors"],
            categorical_cols=[COUNTRY_COL, GENDER_COL, "subject_primary"],
            group_col="paper_code",
        )

        formula = f"Q('{y}') ~ {formula_rhs}"

        try:
            model = smf.ols(
                formula,
                data=model_df
            ).fit(
                cov_type="cluster",
                cov_kwds={"groups": model_df["paper_code"]}
            )
        except Exception as e:
            print(f"Review model failed: {cls} | {prefix} | {e}")
            continue

        for term, est in model.params.items():
            if term == "Intercept":
                continue

            conf = model.conf_int().loc[term]

            rows.append({
                "aspect_class": cls,
                "predictor_type": predictor_type,
                "term": term,
                "estimate": est,
                "conf_low": conf[0],
                "conf_high": conf[1],
                "p_value": model.pvalues.get(term, np.nan),
                "n_reviews": int(model.nobs),
                "n_papers": model_df["paper_code"].nunique(),
                "model": prefix,
            })

    out = pd.DataFrame(rows)

    if len(out) > 0:
        out["p_adj_bh"] = multipletests(out["p_value"].fillna(1), method="fdr_bh")[1]

    save_table(out, OUT_TABLES / f"model_review_aspect_ratio_ols__{prefix}.csv")
    return out


base_review_controls = "publication_year + C(subject_primary) + log_review_word_count + log_n_authors"

review_country_models = fit_review_aspect_ols(
    review_country,
    predictor_type="first_author_country",
    formula_rhs=f"C({COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + {base_review_controls}",
    prefix="country",
)

review_gender_models = fit_review_aspect_ols(
    review_gender,
    predictor_type="first_author_gender",
    formula_rhs=f"C({GENDER_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_review_controls}",
    prefix="gender",
)

review_combined_models = fit_review_aspect_ols(
    review_combined,
    predictor_type="country_and_gender",
    formula_rhs=f"C({COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + C({GENDER_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_review_controls}",
    prefix="country_gender",
)

In [ ]:
# =========================================================
# 6. Paper-level JSD models
# =========================================================

def fit_jsd_model(df: pd.DataFrame, predictor_type: str, formula_rhs: str, prefix: str, y_col: str = JSD_COL) -> pd.DataFrame:
    needed = ["paper_code", y_col, "publication_year", "subject_primary", "n_reviews_valid_for_jsd", "log_aspect_total_count", "log_n_authors"]
    if COUNTRY_COL in formula_rhs:
        needed.append(COUNTRY_COL)
    if GENDER_COL in formula_rhs:
        needed.append(GENDER_COL)
    needed = [c for c in dict.fromkeys(needed) if c in df.columns]

#    model_df = df[needed].dropna().copy()
#    formula = f"{y_col} ~ {formula_rhs}"
#    model = smf.ols(formula, data=model_df).fit(cov_type="HC3")

    model_df = df[needed].dropna().copy()
    model_df = coerce_for_statsmodels(
        model_df,
        numeric_cols=[y_col, "publication_year", "n_reviews_valid_for_jsd", "log_aspect_total_count", "log_n_authors"],
        categorical_cols=[COUNTRY_COL, GENDER_COL, "subject_primary"],
        group_col="paper_code",
    )

    formula = f"{y_col} ~ {formula_rhs}"
    model = smf.ols(formula, data=model_df).fit(cov_type="HC3")

    rows = []
    conf_int = model.conf_int()
    for term, est in model.params.items():
        if term == "Intercept":
            continue
        conf = conf_int.loc[term]
        rows.append({
            "dependent_var": y_col,
            "predictor_type": predictor_type,
            "term": term,
            "estimate": est,
            "conf_low": conf[0],
            "conf_high": conf[1],
            "p_value": model.pvalues.get(term, np.nan),
            "n_papers": int(model.nobs),
            "r_squared": model.rsquared,
            "adj_r_squared": model.rsquared_adj,
            "model": prefix,
        })

    out = pd.DataFrame(rows)
    if len(out) > 0:
        out["p_adj_bh"] = multipletests(out["p_value"].fillna(1), method="fdr_bh")[1]
    save_table(out, OUT_TABLES / f"model_paper_jsd_ols__{prefix}.csv")
    return out

base_jsd_controls = "publication_year + C(subject_primary) + n_reviews_valid_for_jsd + log_aspect_total_count + log_n_authors"

jsd_country = fit_jsd_model(
    paper_country,
    predictor_type="first_author_country",
    formula_rhs=f"C({COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + {base_jsd_controls}",
    prefix="country",
)

jsd_gender = fit_jsd_model(
    paper_gender,
    predictor_type="first_author_gender",
    formula_rhs=f"C({GENDER_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_jsd_controls}",
    prefix="gender",
)

jsd_combined = fit_jsd_model(
    paper_combined,
    predictor_type="country_and_gender",
    formula_rhs=f"C({COUNTRY_COL}, Treatment(reference='{COUNTRY_REFERENCE}')) + C({GENDER_COL}, Treatment(reference='{GENDER_REFERENCE}')) + {base_jsd_controls}",
    prefix="country_gender",
)

jsd_all = pd.concat([jsd_country, jsd_gender, jsd_combined], ignore_index=True)
save_table(jsd_all, OUT_TABLES / "model_paper_jsd_ols__all_main_models.csv")

In [ ]:
# =========================================================
# 7. Per-aspect disagreement on the strict main sample
# =========================================================

def mean_pairwise_abs(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    if n < 2:
        return np.nan
    values = np.sort(values)
    # Efficient sum of pairwise absolute differences:
    # sum_{i<j} (x_j - x_i) = sum_i (2*i - n + 1) * x_i, with i zero-based.
    weights = 2 * np.arange(n) - n + 1
    pair_sum = np.sum(weights * values)
    return float(pair_sum / (n * (n - 1) / 2))


def compute_per_aspect_disagreement(review_df: pd.DataFrame) -> pd.DataFrame:
    temp = review_df.copy()
    for cls in ASPECT_CLASSES:
        temp[f"share__{cls}"] = np.where(
            temp["aspect_total_count"] > 0,
            temp[cls] / temp["aspect_total_count"],
            np.nan,
        )

    rows = []
    for paper_code, g in temp.groupby("paper_code"):
        row = {"paper_code": paper_code, "n_reviews": g["review_id"].nunique()}
        for cls in ASPECT_CLASSES:
            row[f"mapd__{cls}"] = mean_pairwise_abs(g[f"share__{cls}"].values)
        rows.append(row)

    out = pd.DataFrame(rows)
    # Merge paper-level background variables.
    merge_cols = [
        "paper_code",
        COUNTRY_COL,
        GENDER_COL,
        "publication_year",
        "subject_primary",
        "n_authors",
        "log_n_authors",
        "n_reviews_valid_for_jsd",
        "aspect_total_count",
        "log_aspect_total_count",
        JSD_COL,
    ]
    merge_cols = [c for c in merge_cols if c in paper_main.columns]
    out = out.merge(paper_main[merge_cols], on="paper_code", how="left")
    return out

per_aspect = compute_per_aspect_disagreement(review_main)
save_table(per_aspect, OUT_SAMPLES / "main_strict_paper_per_aspect_disagreement.parquet")

per_aspect_summary_rows = []
for cls in ASPECT_CLASSES:
    vals = per_aspect[f"mapd__{cls}"].dropna()
    per_aspect_summary_rows.append({
        "aspect_class": cls,
        "aspect_label": ASPECT_SHORT.get(cls, cls),
        "n_papers": len(vals),
        "mean_mapd": vals.mean(),
        "median_mapd": vals.median(),
        "sd_mapd": vals.std(),
        "q25_mapd": vals.quantile(0.25),
        "q75_mapd": vals.quantile(0.75),
    })
per_aspect_summary = pd.DataFrame(per_aspect_summary_rows).sort_values("mean_mapd", ascending=False)
save_table(per_aspect_summary, OUT_TABLES / "descriptive_per_aspect_disagreement_main_strict.csv")

The next cell writes only aggregate figure-data CSVs. Figure rendering is delegated to notebook 09; no microdata leave the demo results tree.

In [ ]:
# =========================================================
# 8. Final-result figures, English text, no titles
# =========================================================

def aspect_distribution_by_group(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    rows = []
    temp = df.copy()
    temp[group_col] = temp[group_col].apply(clean_value)
    for group, g in temp.groupby(group_col):
        total = g[ASPECT_CLASSES].sum().sum()
        if total <= 0:
            continue
        for cls in ASPECT_CLASSES:
            rows.append({
                "group": group,
                "aspect_class": cls,
                "aspect_label": ASPECT_SHORT.get(cls, cls),
                "aspect_ratio": g[cls].sum() / total,
                "n_papers": g["paper_code"].nunique(),
                "n_reviews": g["review_id"].nunique(),
            })
    return pd.DataFrame(rows)


def make_deviation_matrix(df: pd.DataFrame, group_col: str, groups_to_plot: list[str] | None = None) -> pd.DataFrame:
    dist = aspect_distribution_by_group(df, group_col)
    mat = dist.pivot(index="group", columns="aspect_class", values="aspect_ratio")
    mat = mat[[c for c in ASPECT_CLASSES if c in mat.columns]]

    overall = df[ASPECT_CLASSES].sum()
    overall = overall / overall.sum()
    for cls in mat.columns:
        mat[cls] = mat[cls] - overall[cls]
    mat = mat * 100  # percentage points

    if groups_to_plot is not None:
        mat = mat.loc[[g for g in groups_to_plot if g in mat.index]]
    return mat


country_fig_df = review_country[~review_country[COUNTRY_COL].isin(["Other", "Unknown"])].copy()
country_order = country_fig_df.groupby(COUNTRY_COL)["paper_code"].nunique().sort_values(ascending=False).index.tolist()
country_dev = make_deviation_matrix(country_fig_df, COUNTRY_COL, country_order)
country_dev.to_csv(OUT_TABLES / "figure_data_country_aspect_deviation_pp.csv", encoding="utf-8-sig")
gender_order = [g for g in ["male", "female"] if g in review_gender[GENDER_COL].unique()]
gender_dev = make_deviation_matrix(review_gender, GENDER_COL, gender_order)
gender_dev.to_csv(OUT_TABLES / "figure_data_gender_aspect_deviation_pp.csv", encoding="utf-8-sig")
def extract_group_from_term(term):
    match = re.search(r"\[T\.(.*?)\]", term)
    return match.group(1) if match else term
for name, table, group_col in [("country", jsd_country, COUNTRY_COL), ("gender", jsd_gender, GENDER_COL)]:
    filtered = table[table["term"].str.contains(f"C\\({group_col}", regex=True)].copy()
    filtered["label"] = filtered["term"].map(extract_group_from_term)
    if name == "country":
        filtered = filtered[~filtered["label"].isin(["Other", "Unknown"])]
    else:
        filtered["label"] = filtered["label"].replace({"female": "Female first author"})
    save_table(filtered, OUT_TABLES / f"figure_data_jsd_{name}_coefficients.csv")
